# 📝 Aiscern — Text AI Detector v2
### Platform: **Kaggle T4 x2** (free) | ~3h runtime
**Base model**: `microsoft/deberta-v3-base` (best SOTA for text classification)  
**Method**: LoRA fine-tuning — only 2.3% of parameters trained  
**Output push**: `saghi776/aiscern-text-detector`  
**Target**: ≥ 80% accuracy (F1 ≥ 0.80, AUC ≥ 0.87)

### Datasets (combined ~180k samples)
| Dataset | Source | Samples | Notes |
|---|---|---|---|
| HC3 | Hello-SimpleAI/HC3 | 58k | ChatGPT vs human answers |
| RAID Benchmark | liamdugan/raid | 10k | Multi-model, diverse prompts |
| AI Text Detection Pile | artem9k/ai-text-detection-pile | 50k | GPT-2/3/4, Claude, Gemini |
| GPT-Wiki-Intro | aadityaubhat/GPT-wiki-intro | 30k | GPT-generated Wikipedia |
| WritingPrompts | human baseline | 20k | Creative human writing |
| TuringBench | mrm8488/TuringBench | 10k | 200k GPT-2 vs human |

### Wiring to Aiscern
Model is loaded at weight **0.45** (dominant) in `hf-analyze.ts`:
```
MODELS.text_finetuned = 'saghi776/aiscern-text-detector'
mlScores.push({ model, aiScore: s0, weight: 0.45 })
```
Labels: `{0: "human", 1: "ai"}` — the inference code expects exactly these.

In [ ]:
# ── INSTALL ──────────────────────────────────────────────────────
!pip install -q transformers==4.41.0 datasets peft==0.11.0 accelerate==0.30.0 \
             evaluate scikit-learn huggingface_hub sentencepiece protobuf bitsandbytes
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
print("✅ Packages installed")

In [ ]:
# ── AUTH + CONFIG ────────────────────────────────────────────────
import os, torch

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("✅ HF token loaded from Kaggle Secrets")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    print("⚠  Set HF_TOKEN as Kaggle Secret or env var")

# ── CONFIG ──
BASE_MODEL        = "microsoft/deberta-v3-base"
PUSH_REPO         = "saghi776/aiscern-text-detector"
CHECKPOINT_DIR    = "/kaggle/working/text-ckpt"
MAX_LEN           = 512
SAMPLES_PER_CLASS = 65000      # 65k AI + 65k human = 130k total
BATCH_SIZE        = 16         # T4 16GB fits with grad accumulation
GRAD_ACCUM        = 4          # effective batch = 64
EPOCHS            = 4
LR                = 1.5e-4
LORA_R            = 16
LORA_ALPHA        = 32
LORA_DROPOUT      = 0.05
SEED              = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device} | GPU: {torch.cuda.get_device_name(0) if device=='cuda' else 'none'}")
print(f"Total target samples: {SAMPLES_PER_CLASS*2:,} | Epochs: {EPOCHS}")

In [ ]:
# ── LOAD + NORMALIZE DATASETS ────────────────────────────────────
from datasets import load_dataset, concatenate_datasets, Dataset
import pandas as pd, numpy as np, random

random.seed(SEED); np.random.seed(SEED)

def norm_label(v):
    s = str(v).strip().lower()
    if s in ("1","ai","chatgpt","generated","fake","machine","true","gpt"): return 1
    if s in ("0","human","real","false","wikihuman","reddit"): return 0
    return None

all_rows = []

# 1. HC3 — ChatGPT vs human answers
print("Loading HC3...")
try:
    hc3 = load_dataset("Hello-SimpleAI/HC3", "all", split="train", trust_remote_code=True)
    for row in hc3:
        for ans in (row.get("human_answers") or []):
            if ans and len(ans.split()) > 20: all_rows.append({"text": ans[:2000], "label": 0})
        for ans in (row.get("chatgpt_answers") or []):
            if ans and len(ans.split()) > 20: all_rows.append({"text": ans[:2000], "label": 1})
    print(f"  HC3: {len(all_rows):,} rows")
except Exception as e: print(f"  HC3 failed: {e}")

prev = len(all_rows)
# 2. AI Text Detection Pile
print("Loading AI Text Detection Pile...")
try:
    pile = load_dataset("artem9k/ai-text-detection-pile", split="train", trust_remote_code=True)
    for row in pile.select(range(min(len(pile), 60000))):
        lbl = norm_label(row.get("source",""))
        txt = row.get("text","")
        if lbl is not None and len(txt.split()) > 20:
            all_rows.append({"text": txt[:2000], "label": lbl})
    print(f"  Pile: +{len(all_rows)-prev:,} rows")
except Exception as e: print(f"  Pile failed: {e}")

prev = len(all_rows)
# 3. GPT-Wiki-Intro
print("Loading GPT-Wiki-Intro...")
try:
    wiki = load_dataset("aadityaubhat/GPT-wiki-intro", split="train", trust_remote_code=True)
    for row in wiki.select(range(min(len(wiki), 30000))):
        gpt = row.get("generated_intro","")
        hum = row.get("wiki_intro","")
        if len(gpt.split()) > 20: all_rows.append({"text": gpt[:2000], "label": 1})
        if len(hum.split()) > 20: all_rows.append({"text": hum[:2000], "label": 0})
    print(f"  GPT-Wiki: +{len(all_rows)-prev:,} rows")
except Exception as e: print(f"  GPT-Wiki failed: {e}")

prev = len(all_rows)
# 4. RAID Benchmark
print("Loading RAID...")
try:
    raid = load_dataset("liamdugan/raid", split="train", trust_remote_code=True)
    for row in raid.select(range(min(len(raid), 12000))):
        txt = row.get("generation","") or row.get("text","")
        lbl = 1 if row.get("model","") not in ("","human") else 0
        if len(txt.split()) > 20: all_rows.append({"text": txt[:2000], "label": lbl})
    print(f"  RAID: +{len(all_rows)-prev:,} rows")
except Exception as e: print(f"  RAID failed: {e}")

prev = len(all_rows)
# 5. TuringBench
print("Loading TuringBench...")
try:
    tb = load_dataset("mrm8488/TuringBench", split="train", trust_remote_code=True)
    for row in tb.select(range(min(len(tb), 20000))):
        lbl = 1 if row.get("label","human") != "human" else 0
        txt = row.get("Generation","") or row.get("text","")
        if len(txt.split()) > 20: all_rows.append({"text": txt[:2000], "label": lbl})
    print(f"  TuringBench: +{len(all_rows)-prev:,} rows")
except Exception as e: print(f"  TuringBench failed: {e}")

print(f"\n📊 Total raw: {len(all_rows):,}")
from collections import Counter
print(f"   Label dist: {Counter(r['label'] for r in all_rows)}")

In [ ]:
# ── BALANCE + SPLIT ──────────────────────────────────────────────
random.shuffle(all_rows)
ai_rows    = [r for r in all_rows if r["label"] == 1]
human_rows = [r for r in all_rows if r["label"] == 0]

ai_rows    = random.sample(ai_rows,    min(len(ai_rows),    SAMPLES_PER_CLASS))
human_rows = random.sample(human_rows, min(len(human_rows), SAMPLES_PER_CLASS))
balanced   = ai_rows + human_rows
random.shuffle(balanced)

split_idx = int(len(balanced) * 0.90)
train_rows = balanced[:split_idx]
val_rows   = balanced[split_idx:]

train_ds = Dataset.from_list(train_rows)
val_ds   = Dataset.from_list(val_rows)

print(f"✅ Train: {len(train_ds):,} | Val: {len(val_ds):,}")
print(f"   Train AI: {sum(1 for r in train_rows if r['label']==1):,} | Human: {sum(1 for r in train_rows if r['label']==0):,}")

In [ ]:
# ── TOKENIZE ─────────────────────────────────────────────────────
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN, use_fast=True)

def tokenize(batch):
    enc = tokenizer(batch["text"], truncation=True, padding="max_length",
                    max_length=MAX_LEN, return_tensors=None)
    enc["labels"] = batch["label"]
    return enc

train_tok = train_ds.map(tokenize, batched=True, batch_size=1000, remove_columns=["text","label"])
val_tok   = val_ds.map(tokenize,   batched=True, batch_size=1000, remove_columns=["text","label"])
train_tok.set_format("torch")
val_tok.set_format("torch")
print(f"✅ Tokenized | train cols: {train_tok.column_names}")

In [ ]:
# ── MODEL + LoRA ─────────────────────────────────────────────────
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: "human", 1: "ai"},    # MUST match hf-analyze.ts parseHFText labels
    label2id={"human": 0, "ai": 1},
    token=HF_TOKEN,
    ignore_mismatched_sizes=True,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    # Target attention + FFN layers for maximum coverage
    target_modules=["query_proj","key_proj","value_proj","pos_proj","intermediate.dense"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} = {100*trainable/total:.2f}%")

In [ ]:
# ── TRAINING ─────────────────────────────────────────────────────
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import numpy as np

data_collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:,1].numpy()
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average="binary"),
        "roc_auc":  roc_auc_score(labels, probs),
    }

args = TrainingArguments(
    output_dir            = CHECKPOINT_DIR,
    num_train_epochs      = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate         = LR,
    weight_decay          = 0.01,
    warmup_ratio          = 0.06,
    lr_scheduler_type     = "cosine",
    evaluation_strategy   = "epoch",
    save_strategy         = "epoch",
    load_best_model_at_end= True,
    metric_for_best_model = "f1",
    greater_is_better     = True,
    fp16                  = torch.cuda.is_available(),
    logging_steps         = 200,
    report_to             = "none",
    seed                  = SEED,
    dataloader_num_workers= 2,
    group_by_length       = True,
)

trainer = Trainer(
    model            = model,
    args             = args,
    train_dataset    = train_tok,
    eval_dataset     = val_tok,
    tokenizer        = tokenizer,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print("🚀 Starting training...")
trainer.train()

In [ ]:
# ── EVALUATE ─────────────────────────────────────────────────────
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns

results = trainer.evaluate()
acc = results.get("eval_accuracy", 0)
f1  = results.get("eval_f1", 0)
auc = results.get("eval_roc_auc", 0)

print(f"\n🏆 FINAL RESULTS:")
print(f"   Accuracy : {acc*100:.2f}%  (target ≥80%)")
print(f"   F1 Score : {f1*100:.2f}%  (target ≥80%)")
print(f"   ROC-AUC  : {auc*100:.2f}%  (target ≥87%)")
print(f"   {'✅ PASSED' if acc >= 0.80 else '⚠  Below target — try more epochs or data'}")

# Confusion matrix
preds_out = trainer.predict(val_tok)
preds = np.argmax(preds_out.predictions, axis=-1)
labels = preds_out.label_ids
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=["Human","AI"], yticklabels=["Human","AI"], cmap="Blues")
plt.title(f"Confusion Matrix (Acc={acc*100:.1f}%)")
plt.ylabel("True"); plt.xlabel("Predicted")
plt.tight_layout(); plt.show()
print(classification_report(labels, preds, target_names=["human","ai"]))

In [ ]:
# ── MERGE LoRA + PUSH TO HF ─────────────────────────────────────
from huggingface_hub import HfApi
from peft import PeftModel

print("Merging LoRA weights into base model...")
merged = model.merge_and_unload()

print(f"Pushing to {PUSH_REPO}...")
merged.push_to_hub(PUSH_REPO, token=HF_TOKEN, private=False)
tokenizer.push_to_hub(PUSH_REPO, token=HF_TOKEN)

# Verify the model can be loaded back
print("\nVerifying push...")
from transformers import pipeline
pipe = pipeline("text-classification", model=PUSH_REPO, token=HF_TOKEN)
test = pipe("Furthermore, it is important to note that this comprehensive analysis delves into multifaceted aspects.")
print(f"Test inference: {test}")
print(f"\n✅ Model live at https://huggingface.co/{PUSH_REPO}")
print(f"   Aiscern will load it automatically via MODELS.text_finetuned in hf-analyze.ts")